In [1]:
from collections import defaultdict
import random
from random import Random
import os
import pickle as pkl
from pprint import pprint
import time

import numpy as np
import pandas as pd
# from sklearn.model_selection import train_test_split
from tqdm import tqdm

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

# from rdmc import RDKitMol

import matplotlib.pyplot as plt
%matplotlib inline

In [5]:

# by default, RDKit creates canonical smiles
# https://www.rdkit.org/docs/source/rdkit.Chem.rdmolfiles.html#rdkit.Chem.rdmolfiles.MolToSmiles

# https://stackoverflow.com/questions/3898572/what-are-the-most-common-python-docstring-formats
def canonicalize_smi(smi, remove_Hs=True, sanitize=True, remove_atom_mapping=False):
    """
    Create canonicalized SMILES
    
    """
    params = Chem.SmilesParserParams()
    params.removeHs = remove_Hs
    params.sanitize = sanitize

    # https://www.rdkit.org/docs/source/rdkit.Chem.rdmolfiles.html#rdkit.Chem.rdmolfiles.MolFromSmiles
    # by default, sanitize = True
    # by default, removeHs = True darn
    mol = Chem.MolFromSmiles(smi, params)

    # Remove atom map numbers, otherwise the smiles string is long and non-readable
    if remove_atom_mapping:
        for atom in mol.GetAtoms():
            if atom.HasProp("molAtomMapNumber"):
                atom.ClearProp("molAtomMapNumber")
            # atom.SetAtomMapNum(0)
    
    return Chem.rdmolfiles.MolToSmiles(mol)

# modified from here
# https://github.com/rxn4chemistry/rxnfp/blob/6fd48f4927c2178555cc5d71dbfb225fb178f43c/rxnfp/tokenization.py#L122-L153
def process_reaction(rxn_smi):
    """
    Process and canonicalize reaction SMILES
    """
    reactants, reagents, products = rxn_smi.split(">")

    reactants_c = ".".join(sorted([canonicalize_smi(r, remove_atom_mapping=True) for r in reactants.split(".")]))

    if len(reagents) > 0:
        reagents_c = ".".join(sorted([canonicalize_smi(r, remove_atom_mapping=True) for r in reagents.split(".")]))
    else:
        reagents_c = ''

    products_c = ".".join(sorted([canonicalize_smi(p, remove_atom_mapping=True) for p in products.split(".")]))

    return f"{reactants_c}>{reagents_c}>{products_c}"

In [4]:
path = '../data/ccsdtf12/original_v1.0.1/ccsdtf12_dz.csv'
df = pd.read_csv(path)
df

,idx,rsmi,psmi,rinchi,pinchi,dE0,dHrxn298,rmg_family
0,0,[C:1]([c:2]1[n:3][o:4][n:5][n:6]1)([H:7])([H:8...,[C:1]([C:2]([N:3]=[O:4])=[N+:6]=[N-:5])([H:7])...,InChI=1S/C2H3N3O/c1-2-3-5-6-4-2/h1H3,InChI=1S/C2H3N3O/c1-2(4-3)5-6/h1H3,48.61085,26.77621,NaN
1,1,[C:1]([c:2]1[n:3][o:4][n:5][n:6]1)([H:7])([H:8...,[C:1]([N:3]=[C:2]=[N:6][N:5]=[O:4])([H:7])([H:...,InChI=1S/C2H3N3O/c1-2-3-5-6-4-2/h1H3,InChI=1S/C2H3N3O/c1-3-2-4-5-6/h1H3,74.02980,28.79099,NaN
2,2,[C:1]([O:2][C:3]([C:4]([O:5][H:13])([H:11])[H:...,[C:1]1([H:6])([H:7])[O:2][C:3]([H:9])([H:10])[...,"InChI=1S/C3H8O2/c1-5-3-2-4/h4H,2-3H2,1H3",InChI=1S/C3H6O.H2O/c1-2-4-3-1;/h1-3H2;1H2,97.42200,12.60220,NaN
3,3,[C:1]([O:2][C:3]([C:4]([O:5][H:13])([H:11])[H:...,[C:1]([O:2][H:13])([H:6])([H:7])[H:8].[C:3]1([...,"InChI=1S/C3H8O2/c1-5-3-2-4/h4H,2-3H2,1H3","InChI=1S/C2H4O.CH4O/c1-2-3-1;1-2/h1-2H2;2H,1H3",75.25375,28.98589,NaN
4,4,[C:1]([O:2][C:3]([C:4]([O:5][H:13])([H:11])[H:...,[C:1]([O:2][H:13])([H:6])([H:7])[H:8].[C:3]([C...,"InChI=1S/C3H8O2/c1-5-3-2-4/h4H,2-3H2,1H3","InChI=1S/C2H4O.CH4O/c1-2-3;1-2/h2H,1H3;2H,1H3",72.16356,1.41779,NaN
...,...,...,...,...,...,...,...,...
11921,11956,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1]([C:2][C:4]([O:5][C:6](=[O:7])[H:15])([H:...,"InChI=1S/C4H8O3/c1-4(6)2-7-3-5/h3-4,6H,2H2,1H3...","InChI=1S/C4H6O2.H2O/c1-2-3-6-4-5;/h4H,3H2,1H3;1H2",75.56813,79.63518,NaN
11922,11957,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1]([C@@:2]1([H:11])[O:3][C@:6]([O:7][H:12])...,"InChI=1S/C4H8O3/c1-4(6)2-7-3-5/h3-4,6H,2H2,1H3...","InChI=1S/C4H8O3/c1-3-2-6-4(5)7-3/h3-5H,2H2,1H3...",42.41621,5.79695,NaN
11923,11958,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1]([C@@:2]([O:3][H:12])([C:4](=[O:5])[H:14]...,"InChI=1S/C4H8O3/c1-4(6)2-7-3-5/h3-4,6H,2H2,1H3...","InChI=1S/C3H6O2.CH2O/c1-3(5)2-4;1-2/h2-3,5H,1H...",72.75039,30.54744,NaN
11924,11959,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1](=[C:2]([C:4]([O:5][C:6](=[O:7])[H:15])([...,"InChI=1S/C4H8O3/c1-4(6)2-7-3-5/h3-4,6H,2H2,1H3...","InChI=1S/C4H6O2.H2O/c1-2-3-6-4-5;/h2,4H,1,3H2;1H2",65.83112,14.48350,"1,3_Insertion_ROR"


# Create canonicalized reaction smiles with Hs
- include reverse reactions

In [6]:
cols = ['idx', 'rxn_smiles', 'dE0', 'dHrxn298']
df_new = pd.DataFrame(np.zeros((0, len(cols))), columns=cols)

# i is the pandas index
for i, row in df.iterrows():
    # forward reaction
    rxn_smi = f'{row.rsmi}>>{row.psmi}'
    rxn_smi_cannonical = process_reaction(rxn_smi)
    tmp = np.array([row.idx, rxn_smi_cannonical, row.dE0, row.dHrxn298]).reshape(1, 4)
    df_new = df_new.append(pd.DataFrame(tmp, columns=cols))
    
    
    # append reverse reaction
    rxn_smi = f'{row.psmi}>>{row.rsmi}'
    rxn_smi_cannonical = process_reaction(rxn_smi)
    tmp = np.array([row.idx, rxn_smi_cannonical, row.dE0 - row.dHrxn298, -row.dHrxn298]).reshape(1, 4)
    df_new = df_new.append(pd.DataFrame(tmp, columns=cols))

df_new.idx = df_new.idx.astype(int)
df_new.dE0 = df_new.dE0.astype(float)
df_new.dHrxn298 = df_new.dHrxn298.astype(float)
df_new

KeyboardInterrupt: 

In [28]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 23852 entries, 0 to 0
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   idx         23852 non-null  int64  
 1   rxn_smiles  23852 non-null  object 
 2   dE0         23852 non-null  float64
 3   dHrxn298    23852 non-null  float64
dtypes: float64(2), int64(1), object(1)
memory usage: 931.7+ KB


# Create splits

In [44]:
def create_new_indices(old_indices):
    new_indices = np.zeros(len(old_indices)*2) 
    for i, index in enumerate(old_indices):
        new_indices[2*i] = int(index*2)
        new_indices[2*i + 1] = int(index*2 +1)
    
    return new_indices.astype(int)

## Create random split the wrong way
- randomly shuffle the new data that DOES include adding the reverse reactions

In [60]:
seed = 0   # random seed for reproducability
sizes=(0.85, 0.05, 0.1)  # train, val, test sizes
total_num_datapoints = len(df_new)
print(f'Using {total_num_datapoints} datapoints...')

random = Random(seed)
indices = list(range(total_num_datapoints))
random.shuffle(indices)

train_size = int(sizes[0] * total_num_datapoints)
train_val_size = int((sizes[0] + sizes[1]) * total_num_datapoints)

train = indices[:train_size]
val = indices[train_size:train_val_size]
test = indices[train_val_size:]

Using 23852 datapoints...


In [61]:
indices = [[train_indices_new, val_indices_new, test_indices_new]]
with open(f'../data/canonicalized_smiles/ccsdtf12_random_split_seed_{seed}_{len(train_indices_new)}.pkl', 'wb') as f:
    pkl.dump(indices, f)

## Create random split the correct way
- randomly shuffle the original data. do NOT use the data after adding the reverse since that is useless

In [55]:
seed = 0   # random seed for reproducability
sizes=(0.85, 0.05, 0.1)  # train, val, test sizes
total_num_datapoints = len(df)
print(f'Using {total_num_datapoints} datapoints...')

random = Random(seed)
indices = list(range(total_num_datapoints))
random.shuffle(indices)

train_size = int(sizes[0] * total_num_datapoints)
train_val_size = int((sizes[0] + sizes[1]) * total_num_datapoints)

train = indices[:train_size]
val = indices[train_size:train_val_size]
test = indices[train_val_size:]

Using 11926 datapoints...


In [56]:
# multiply by 2
train_indices_new = create_new_indices(train)
val_indices_new = create_new_indices(val)
test_indices_new = create_new_indices(test)

print(f'len(val): {len(val)}')
print(f'len(val_indices_new): {len(val_indices_new)}')

len(val): 596
len(val_indices_new): 1192


In [57]:
# verify that doubling takes the correct indices
df_new.iloc[val_indices_new, :]

,idx,rxn_smiles,dE0,dHrxn298
0,682,[H]c1c(C([H])([H])[H])c([H])n([H])c1[H]>>[H]C1...,84.89180,71.91494
0,682,[H]C1=[C-][N+]([H])([H])C([H])=C1C([H])([H])[H...,12.97686,-71.91494
0,6237,[H]C1=C([H])C([H])([H])C(=O)N1[H]>>[H]/N=C1\OC...,117.25221,16.52157
0,6237,[H]/N=C1\OC([H])=C([H])C1([H])[H]>>[H]C1=C([H]...,100.73064,-16.52157
0,4860,[H]/N=C(\[H])N(C([H])=O)C([H])([H])[H]>>[H]C(=...,43.47672,9.72335
...,...,...,...,...
0,5153,[H]C(=C=O)C([H])([H])C([H])([H])[H].[H]C([H])(...,65.30747,-24.47287
0,8958,[H]OC([H])([H])[C@@]1([H])N(C([H])([H])C([H])(...,83.24765,71.79146
0,8958,[H]OC([H])([H])[C@]1([H])C([H])=[N+]1[C-]([H])...,11.45619,-71.79146
0,7889,[H]O/N=C(\C([H])([H])[H])[C@]1([H])OC1([H])[H]...,77.92283,15.09075


In [59]:
indices = [[train_indices_new, val_indices_new, test_indices_new]]
with open(f'../data/canonicalized_smiles/ccsdtf12_random_split_using_fwd_reactants_seed_{seed}_{len(train_indices_new)}.pkl', 'wb') as f:
    pkl.dump(indices, f)

## Create Scaffold Split

In [62]:
def str_to_mol(string, explicit_hydrogens=False):
    """
    Converts an InChI or SMILES string to an RDKit molecule.

    :param string: The InChI or SMILES string.
    :param explicit_hydrogens: Whether to treat hydrogens explicitly.
    :return: The RDKit molecule.
    """
    RDKIT_SMILES_PARSER_PARAMS = Chem.SmilesParserParams()
    if string.startswith('InChI'):
        mol = Chem.MolFromInchi(string, removeHs=not explicit_hydrogens)
    else:
        # Set params here so we don't remove hydrogens with atom mapping
        RDKIT_SMILES_PARSER_PARAMS.removeHs = not explicit_hydrogens
        mol = Chem.MolFromSmiles(string, RDKIT_SMILES_PARSER_PARAMS)

    if explicit_hydrogens:
        return Chem.AddHs(mol)
    else:
        return Chem.RemoveHs(mol)

In [63]:
def generate_scaffold(mol, include_chirality=False):
    """
    Compute the Bemis-Murcko scaffold for a SMILES string.

    :param mol: A smiles string or an RDKit molecule.
    :param include_chirality: Whether to include chirality.
    :return:
    """
    mol = str_to_mol(mol) if type(mol) == str else mol
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=include_chirality)

    return scaffold

In [64]:
def scaffold_to_smiles(mols,
                       use_indices=False):
    """
    Computes scaffold for each smiles string and returns a mapping from scaffolds to sets of smiles.

    :param mols: A list of smiles strings or RDKit molecules.
    :param use_indices: Whether to map to the smiles' index in all_smiles rather than mapping
    to the smiles string itself. This is necessary if there are duplicate smiles.
    :return: A dictionary mapping each unique scaffold to all smiles (or smiles indices) which have that scaffold.
    """
    scaffolds = defaultdict(set)
    for i, mol in tqdm(enumerate(mols), total=len(mols)):
        scaffold = generate_scaffold(mol)
        if use_indices:
            scaffolds[scaffold].add(i)
        else:
            scaffolds[scaffold].add(mol)

    return scaffolds

In [65]:
# modify this function to just return the indices
def scaffold_split(mols,
                   sizes=(0.8, 0.1, 0.1),
                   balanced=False,
                   seed=0,
                   ):
    """
    Split a dataset by scaffold so that no molecules sharing a scaffold are in the same split.

    :param data: A MoleculeDataset or ReactionDataset.
    
    :param mols: a list of rdkit molecules
    
    :param sizes: A length-3 tuple with the proportions of data in the
    train, validation, and test sets.
    :param balanced: Try to balance sizes of scaffolds in each set, rather than just putting smallest in test set.
    :param seed: Seed for shuffling when doing balanced splitting.
    :param logger: A logger.
    :return: A tuple containing the train, validation, and test splits of the data.
    """
    assert sum(sizes) == 1

    # Split
    train_size, val_size, test_size = sizes[0] * len(mols), sizes[1] * len(mols), sizes[2] * len(mols)
    train, val, test = [], [], []
    train_scaffold_count, val_scaffold_count, test_scaffold_count = 0, 0, 0

    # Map from scaffold to index in the data
    # Only use reactant molecules for reaction datasets
    # mols = list(zip(*data.mols()))[0] if isinstance(data, ReactionDataset) else data.mols()
    
    scaffold_to_indices = scaffold_to_smiles(mols, use_indices=True)

    if balanced:  # Put stuff that's bigger than half the val/test size into train, rest just order randomly
        index_sets = list(scaffold_to_indices.values())
        big_index_sets = []
        small_index_sets = []
        for index_set in index_sets:
            if len(index_set) > val_size / 2 or len(index_set) > test_size / 2:
                big_index_sets.append(index_set)
            else:
                small_index_sets.append(index_set)
        random.seed(seed)
        random.shuffle(big_index_sets)
        random.shuffle(small_index_sets)
        index_sets = big_index_sets + small_index_sets
    else:  # Sort from largest to smallest scaffold sets
        index_sets = sorted(list(scaffold_to_indices.values()),
                            key=lambda index_set: len(index_set),
                            reverse=True)

    for index_set in index_sets:
        if len(train) + len(index_set) <= train_size:
            train += index_set
            train_scaffold_count += 1
        elif len(val) + len(index_set) <= val_size:
            val += index_set
            val_scaffold_count += 1
        else:
            test += index_set
            test_scaffold_count += 1

    
    print(f'Total scaffolds = {len(scaffold_to_indices):,} | '
          f'train scaffolds = {train_scaffold_count:,} | '
          f'val scaffolds = {val_scaffold_count:,} | '
          f'test scaffolds = {test_scaffold_count:,}')
    
    # log_scaffold_stats(data, index_sets, logger=logger)
    
    return train, val, test
    
#     # Map from indices to data
#     train = [data[i] for i in train]
#     val = [data[i] for i in val]
#     test = [data[i] for i in test]

#     if isinstance(data, ReactionDataset):
#         return ReactionDataset(train), ReactionDataset(val), ReactionDataset(test)
#     else:
#         return MoleculeDataset(train), MoleculeDataset(val), MoleculeDataset(test)

In [67]:
mols = [str_to_mol(rsmi, explicit_hydrogens=True) for rsmi in df.rsmi]
len(mols)

11926

In [70]:
seed = 0  # 0 3 5 6 42
train_indices, val_indices, test_indices = scaffold_split(mols,
                                                          sizes=(0.85, 0.05, 0.1),
                                                          balanced=True,
                                                          seed=seed)
len(train_indices), len(val_indices), len(test_indices)

100%|██████████| 11926/11926 [00:01<00:00, 10671.68it/s]

Total scaffolds = 465 | train scaffolds = 396 | val scaffolds = 24 | test scaffolds = 45


(10137, 596, 1193)

In [73]:
# multiply by 2
train_indices_new = create_new_indices(train_indices)
val_indices_new = create_new_indices(val_indices)
test_indices_new = create_new_indices(test_indices)

print(f'len(val): {len(val_indices)}')
print(f'len(val_indices_new): {len(val_indices_new)}')

len(val): 596
len(val_indices_new): 1192


In [74]:
# verify that doubling takes the correct indices
df_new.iloc[val_indices_new, :]

,idx,rxn_smiles,dE0,dHrxn298
0,3604,[H]N1C([H])([H])[C@]1([H])C#N>>[H]N=[C-][C@@]1...,132.16551,81.94302
0,3604,[H]N=[C-][C@@]1([H])[C+]([H])N1[H]>>[H]N1C([H]...,50.22249,-81.94302
0,3605,[H]N1C([H])([H])[C@]1([H])C#N>>[H]N1[C][C@]2([...,121.73688,72.01887
0,3605,[H]N1[C][C@]2([H])N([H])[C@]12[H]>>[H]N1C([H])...,49.71801,-72.01887
0,3606,[H]N1C([H])([H])[C@]1([H])C#N>>[H][C]N([H])C([...,74.70348,29.45022
...,...,...,...,...
0,6489,[H]N([H])C#CC([H])([H])[H].[N-]=[N+]=O>>[H]N([...,28.57565,-41.98517
0,10096,[H]N([H])c1nnc(N([H])[H])o1>>[H]/N=C1\N=N[C@](...,97.49959,20.61839
0,10096,[H]/N=C1\N=N[C@]([H])(N([H])[H])O1>>[H]N([H])c...,76.88120,-20.61839
0,10097,[H]N([H])c1nnc(N([H])[H])o1>>[H]/N=c1\oc(N([H]...,58.34083,5.46904


In [75]:
indices = [[train_indices_new, val_indices_new, test_indices_new]]
with open(f'../data/canonicalized_smiles/ccsdtf12_scaffold_split_using_fwd_reactants_seed_{seed}_{len(train_indices_new)}.pkl', 'wb') as f:
    pkl.dump(indices, f)

In [28]:
# df_new.iloc[splits[0][0], :].to_csv('ccsdtf12_dz_bac_fwd_rev_chemprop_ea_train.csv', index=False)
# df_new.iloc[splits[0][1], :].to_csv('ccsdtf12_dz_bac_fwd_rev_chemprop_ea_val.csv', index=False)
# df_new.iloc[splits[0][2], :].to_csv('ccsdtf12_dz_bac_fwd_rev_chemprop_ea_test.csv', index=False)